# 🔬 AI-Based Picture Text Mining & Translation System
## Google Colab Setup Notebook

Run each cell in order. Every cell shows a **✅ / ❌** status.

### 📋 API Keys Setup (Groq & Sarvam)

**Groq API Key (For AI Correction):**
1. Go to **[console.groq.com](https://console.groq.com)**
2. Sign up — **no credit card required**
3. Navigate to **API Keys** section and click **Create API Key**

**Sarvam AI API Key (For Indian Language Translation):**
1. Go to **[dashboard.sarvam.ai](https://dashboard.sarvam.ai/)**
2. Create an account and navigate to API Keys to generate one.

For **AWS EC2** add to `~/.bashrc`:
```bash
export GROQ_API_KEY="your_groq_key"
export SARVAM_API_KEY="your_sarvam_key"
```

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 — GPU CHECK
# ═══════════════════════════════════════════════
import subprocess, os

def check_gpu():
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                                 '--format=csv,noheader'], capture_output=True, text=True)
        if result.returncode == 0 and result.stdout.strip():
            gpu_info = result.stdout.strip()
            print(f'✅ GPU detected: {gpu_info}')
            return True
    except FileNotFoundError:
        pass
    print('❌ No GPU detected! Go to Runtime → Change runtime type → GPU (T4)')
    return False

GPU_AVAILABLE = check_gpu()

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 — INSTALL ALL PACKAGES
# ═══════════════════════════════════════════════
import subprocess, sys

packages = [
    'groq',
    'opencv-python-headless',
    'pytesseract',
    'easyocr',
    'transformers',
    'torch torchvision',
    'paddlepaddle paddleocr',
    'python-doctr[torch]',
    'Pillow',
    'scikit-image',
    'imutils',
    'langdetect',
    'fasttext-wheel',
    'pandas',
    'scikit-learn',
    'nltk',
    'matplotlib',
    'seaborn',
    'streamlit',
    'pyngrok',
    'sentencepiece',
    'googletrans==4.0.0-rc1',
    'requests',
    'numpy',
]

results = {}
for pkg in packages:
    name = pkg.split()[0].split('=')[0].split('[')[0]
    ret = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(),
                         capture_output=True, text=True)
    ok = ret.returncode == 0
    results[name] = ok
    status = '✅' if ok else '❌'
    print(f'{status} {name}')

failed = [k for k, v in results.items() if not v]
if failed:
    print(f'\n⚠️  Failed packages: {", ".join(failed)}')
else:
    print(f'\n🎉 All {len(results)} packages installed successfully!')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3 — INSTALL TESSERACT + INDIAN LANGUAGES
# ═══════════════════════════════════════════════
import subprocess

lang_packs = [
    'tesseract-ocr',
    'tesseract-ocr-hin',   # Hindi
    'tesseract-ocr-tam',   # Tamil
    'tesseract-ocr-tel',   # Telugu
    'tesseract-ocr-ben',   # Bengali
    'tesseract-ocr-kan',   # Kannada
    'tesseract-ocr-mal',   # Malayalam
    'tesseract-ocr-guj',   # Gujarati
]

ret = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
for pkg in lang_packs:
    ret = subprocess.run(['apt-get', 'install', '-y', '-qq', pkg],
                         capture_output=True, text=True)
    status = '✅' if ret.returncode == 0 else '❌'
    print(f'{status} {pkg}')

# Verify
ret = subprocess.run(['tesseract', '--list-langs'], capture_output=True, text=True)
if ret.returncode == 0:
    langs = ret.stdout.strip().split('\n')[1:]  # skip header
    print(f'\n✅ Tesseract installed with {len(langs)} languages: {", ".join(langs)}')
else:
    print('❌ Tesseract installation failed')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4 — MOUNT GOOGLE DRIVE
# ═══════════════════════════════════════════════
import os
from pathlib import Path

IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/OCR_Project')
else:
    BASE = Path('.')

dirs = ['dataset/images', 'outputs/charts', 'cache', 'modules', 'config']
for d in dirs:
    (BASE / d).mkdir(parents=True, exist_ok=True)
    print(f'✅ {d}/')

print(f'\n📂 Project root: {BASE}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5 — SET API KEYS
# ═══════════════════════════════════════════════
import os
from getpass import getpass

# 1. Groq API Key
if not os.environ.get('GROQ_API_KEY'):
    key = getpass('🔑 Enter your Groq API Key: ')
    os.environ['GROQ_API_KEY'] = key

masked_groq = os.environ['GROQ_API_KEY'][:8] + '...' + os.environ['GROQ_API_KEY'][-4:]
print(f'✅ Groq API Key set: {masked_groq}')

# 2. Sarvam API Key
if not os.environ.get('SARVAM_API_KEY'):
    key2 = getpass('🔑 Enter your Sarvam AI API Key (Press Enter to skip if you don\'t have one): ')
    if key2.strip():
        os.environ['SARVAM_API_KEY'] = key2.strip()
        print(f'✅ Sarvam API Key set: {key2[:4]}...{key2[-4:]}')
    else:
        print('⚠️  Sarvam API Key skipped! Indian language translation might fallback to Googletrans.')


In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 — VERIFY GROQ API CONNECTION
# ═══════════════════════════════════════════════
import os

try:
    from groq import Groq
    client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': 'Reply with OK'}],
        max_tokens=5,
        temperature=0,
    )
    reply = response.choices[0].message.content.strip()
    print(f'✅ Groq API ping successful!  Model replied: "{reply}"')
    print(f'   Model      : {response.model}')
    print(f'   Tokens used: {response.usage.total_tokens}')
except Exception as e:
    print(f'❌ Groq API connection failed: {e}')
    print('   Check your API key and internet connection.')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 7 — VERIFY ALL IMPORTS
# ═══════════════════════════════════════════════
checks = {
    'OpenCV':       lambda: __import__('cv2'),
    'PyTesseract':  lambda: __import__('pytesseract'),
    'EasyOCR':      lambda: __import__('easyocr'),
    'Transformers': lambda: __import__('transformers'),
    'PyTorch':      lambda: __import__('torch'),
    'PaddleOCR':    lambda: __import__('paddleocr'),
    'Pillow':       lambda: __import__('PIL'),
    'scikit-image': lambda: __import__('skimage'),
    'langdetect':   lambda: __import__('langdetect'),
    'Pandas':       lambda: __import__('pandas'),
    'NLTK':         lambda: __import__('nltk'),
    'Matplotlib':   lambda: __import__('matplotlib'),
    'Seaborn':      lambda: __import__('seaborn'),
    'Streamlit':    lambda: __import__('streamlit'),
    'Groq SDK':     lambda: __import__('groq'),
}

ok_count = 0
for name, test in checks.items():
    try:
        test()
        print(f'✅ {name}')
        ok_count += 1
    except Exception as e:
        print(f'❌ {name}: {e}')

print(f'\n{"🎉 All" if ok_count == len(checks) else "⚠️  " + str(ok_count) + "/" + str(len(checks))} checks passed')

---
### ✅ Setup Complete!
All dependencies installed, Tesseract configured, Drive mounted, and APIs verified.

**Next:** Run `!streamlit run app.py &` or import the pipeline module.